# 11 · Async consumers and connection recovery

**Level:** advanced · **Time:** 35 minutes · **Prerequisites:** 00, 07, and 10; basic `async` / `await`.

Use `aio-pika` to overlap waiting work without sharing a Pika blocking connection across threads. Then optionally restart the local broker while a robust connection is active.

Jupyter already has an event loop: use top-level `await`, not `asyncio.run(...)` inside a notebook. A robust connection can restore declared topology and consumers after reconnecting; it does not make an interrupted publish atomic or remove duplicates.


In [ ]:
from pathlib import Path
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "course" / "rabbit.py").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from course.rabbit import connect, unique_name, wait_for_message, broker_url
import pika
print("Course root:", ROOT)


## Step 1 — Publish and process four jobs asynchronously

We bound connection setup and the whole demo. `message.process()` acknowledges when its block exits normally. Here every task succeeds; a real failure needs the retry/parking policy from 08. Waiting uses `asyncio.sleep` so other tasks can run. CPU-heavy work should use an appropriate process pool or worker system.


In [ ]:
import asyncio
import aio_pika
import time

async def async_demo():
    connection = await aio_pika.connect_robust(broker_url(), timeout=5, reconnect_interval=1)
    async with connection:
        channel = await connection.channel(publisher_confirms=True)
        await channel.set_qos(prefetch_count=4)
        queue = await channel.declare_queue(unique_name("async"), exclusive=True, auto_delete=True)
        try:
            for number in range(4):
                await channel.default_exchange.publish(
                    aio_pika.Message(body=str(number).encode()), routing_key=queue.name, mandatory=True,
                )
            completed = []
            async def handle(message):
                async with message.process(requeue=False):
                    await asyncio.sleep(0.15)
                    completed.append(int(message.body))
            started = time.perf_counter()
            async with queue.iterator() as iterator:
                # Keep all acknowledgements in this event loop; no thread shares this channel.
                messages = [await asyncio.wait_for(iterator.__anext__(), timeout=5) for _ in range(4)]
                await asyncio.gather(*(handle(message) for message in messages))
            assert sorted(completed) == [0, 1, 2, 3]
            print("Completed:", completed, "elapsed:", round(time.perf_counter() - started, 3), "seconds")
        finally:
            # auto_delete may already remove the queue when its last consumer cancels.
            if not channel.is_closed:
                await channel.close()

await asyncio.wait_for(async_demo(), timeout=20)


## Step 2 — Optional live recovery experiment

The default flag is false, so **Run All** does not disrupt services or wait for a human. To demonstrate recovery:

1. Finish other lessons and keep a terminal ready at the course root.
2. Set `RUN_RECOVERY_LAB = True` and run the next cell.
3. When it prints the prompt, run `docker compose restart rabbitmq` in the terminal.
4. The cell allows 45 seconds for a reconnect, then verifies a persistent message remains available. If your machine starts slowly, increase that explicit timeout.

This deliberately affects the shared local lab broker. No restart command is executed by notebook code. The queue is durable so the pre-restart publication has a chance to survive with the preserved volume. Restoration depends on the broker becoming reachable again with compatible topology and permissions.


In [ ]:
RUN_RECOVERY_LAB = False

async def recovery_demo():
    reconnected = asyncio.Event()
    connection = await aio_pika.connect_robust(broker_url(), timeout=5, reconnect_interval=1)
    async def record_reconnect(sender, **kwargs):
        reconnected.set()
    connection.reconnect_callbacks.add(record_reconnect)
    queue = None
    try:
        channel = await connection.channel(publisher_confirms=True)
        queue = await channel.declare_queue(unique_name("recovery"), durable=True)
        await channel.default_exchange.publish(
            aio_pika.Message(body=b"survive-restart", delivery_mode=aio_pika.DeliveryMode.PERSISTENT),
            routing_key=queue.name, mandatory=True,
        )
        print("Recovery queue:", queue.name, flush=True)
        print("Now run in the terminal: docker compose restart rabbitmq", flush=True)
        await asyncio.wait_for(reconnected.wait(), timeout=45)
        message = await queue.get(timeout=5, fail=True)
        async with message.process():
            assert message.body == b"survive-restart"
        print("Reconnected and received the pre-restart message")
    finally:
        try:
            if queue is not None and connection.connected.is_set():
                await asyncio.wait_for(queue.delete(if_unused=False, if_empty=False), timeout=5)
        finally:
            await connection.close()

if RUN_RECOVERY_LAB:
    await asyncio.wait_for(recovery_demo(), timeout=60)
else:
    print("Recovery drill skipped. Set RUN_RECOVERY_LAB=True to perform it manually.")


## Exercise — Reconnection is not replay

What should you do when the network drops during `publish()` and you do not know whether the broker confirmed it? What happens to work that was in progress during disconnection?

**Suggested answer:** retain a stable message ID, retry according to a bounded policy, and tolerate duplicates at the consumer. A robust client restores connectivity, not arbitrary application state. An unacknowledged delivery may return to another consumer; partially completed effects still require idempotency. Decide whether to cancel in-flight work on loss and how to reconcile uncertain outcomes.

**Troubleshooting:** `RuntimeError: asyncio.run() cannot be called...` means the notebook event loop is already running. A recovery timeout can mean the broker never stopped, never recovered, or the vhost/credentials changed. If cleanup cannot reach the broker, delete only the printed `rmqcourse.recovery...` queue after recovery.

**Cleanup:** the normal demo's exclusive queue is removed when the connection closes, even if the demo fails before registering a consumer; it also auto-deletes after its consumer cancels. The optional demo explicitly deletes its durable queue after recovery.

## References

- [aio-pika quick start](https://docs.aio-pika.com/quick-start.html)
- [aio-pika robust connection options](https://docs.aio-pika.com/url-parameters.html)
- [RabbitMQ reliability](https://www.rabbitmq.com/docs/4.2/reliability)
